# BTC hourly feature sanity check — step 084

Inspect stored features against actual BTC bars: price/mean distance, returns,
volatility, volume, taker activity, and availability after a data gap.
This is a descriptive check, not an alpha evaluation or a profitability claim.
Only the configured **H-HOURLY TRAIN** period is loaded; no labels or holdout data
are read. Features retain the history used by the writer, including history before TRAIN.

From the repository root in WSL, install with `uv sync --group notebooks` (using
the documented external venv). Select that environment's Python kernel in your
notebook editor, then **Run All**. If data are absent, first run
`uv run python scripts/build_features.py --interval 1h --symbol BTCUSDT`.
The notebook finds the repository from either its own directory or the project root.
Source outputs are cleared; executed copies and PNGs belong under ignored `reports_out/`.


In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from matplotlib.colors import ListedColormap

from helios.data.loader import load_bars, to_micros

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/splits.yaml").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run from the HELIOS repository or research/notebooks directory.")
SYMBOL = "BTCUSDT"
INTERVAL = "1h"
FEATURE_SET = "v1"
PLOT_START, PLOT_END = "2021-01-01", "2021-01-31"
GAP_START, GAP_END = "2018-02-06", "2018-02-18"
STORE = ROOT / f"data/processed/features/feature_set={FEATURE_SET}/interval={INTERVAL}"
STORE = STORE / f"symbol={SYMBOL}"
BARS = ROOT / "data/processed/bars"
OUTPUT = ROOT / "reports_out/feature_check"
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

train = yaml.safe_load((ROOT / "configs/splits.yaml").read_text())["families"]["H-HOURLY"]["train"]
train_start = pd.Timestamp(train["start"], tz="UTC")
train_end = pd.Timestamp(train["end"], tz="UTC") + pd.Timedelta(days=1)
for start, end in [(PLOT_START, PLOT_END), (GAP_START, GAP_END)]:
    if not train_start <= pd.Timestamp(start, tz="UTC") <= pd.Timestamp(end, tz="UTC") < train_end:
        raise ValueError("Inspection windows must remain inside H-HOURLY TRAIN.")

## Verify lineage and load TRAIN

Verify hashes of the partitions consumed and their source bars before reading them.
A changed or incomplete build fails explicitly. The table identifies the original
feature build, including its dirty flag; an inspection does not create new lineage
for that build. Files outside TRAIN are neither loaded nor hashed here.


In [ ]:
manifest = json.loads((STORE / "manifest.json").read_text())
if manifest["symbol"] != SYMBOL or manifest["config"]["interval"] != INTERVAL:
    raise ValueError("Feature manifest does not match the requested symbol/interval.")
if manifest["config"]["feature_set_version"] != FEATURE_SET:
    raise ValueError("Feature set version mismatch.")


def verify(path, expected):
    with path.open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != expected:
        raise ValueError(f"Hash mismatch: {path}; rebuild the feature store.")


years = range(train_start.year, (train_end - pd.Timedelta(days=1)).year + 1)
files_by_path = {entry["path"]: entry for entry in manifest["files"]}
frames = []
for year in years:
    relative = f"year={year}/features.parquet"
    if relative not in files_by_path:
        raise FileNotFoundError(f"Missing TRAIN partition in manifest: {relative}")
    entry = files_by_path[relative]
    verify(STORE / relative, entry["sha256"])
    frames.append(pd.read_parquet(STORE / relative))
    source_path = f"source=binance/interval={INTERVAL}/symbol={SYMBOL}/year={year}/bars.parquet"
    source = next(e for e in manifest["input_snapshot"]["files"] if e["path"] == source_path)
    verify(BARS / source_path, source["sha256"])

features = pd.concat(frames, ignore_index=True).sort_values("open_time")
first, last = to_micros(train["start"]), to_micros(train["end"], end=True)
features = features.loc[features["open_time"].between(first, last)].copy()
if features.empty or features["open_time"].duplicated().any():
    raise ValueError("Expected nonempty, unique feature timestamps.")
bars = load_bars(SYMBOL, INTERVAL, train["start"], train["end"], out_dir=BARS)
np.testing.assert_array_equal(features["open_time"].to_numpy(), bars["open_time"].to_numpy())
specs = pd.DataFrame(manifest["config"]["features"]).set_index("name")
names = specs.index.tolist()
prefix = manifest["availability_prefix"]
for name in names:
    mask = features[prefix + name]
    assert mask.dtype == bool, f"{name}: mask must be boolean"
    assert np.isfinite(features.loc[mask, name]).all(), f"{name}: available nonfinite value"
    assert features.loc[~mask, name].isna().all(), f"{name}: unavailable value is not NaN"
for table in (features, bars):
    table.index = pd.to_datetime(table["open_time"], unit="us", utc=True)
assert features.loc[features[prefix + "taker_buy_ratio"], "taker_buy_ratio"].between(0, 1).all()
for name in ["volatility_24", "volatility_168", "parkinson_24", "garman_klass_24"]:
    assert (features.loc[features[prefix + name], name] >= 0).all(), name
display(pd.Series(manifest["lineage"], name="Feature build lineage").to_frame())
display(specs[["units", "lookback", "warmup"]])
print(f"Verified {len(years)} feature/source partition pairs; {len(features):,} TRAIN rows.")

## Main inspection window

All feature lines apply their availability masks. Temporary hourly plot grids insert
NaN at missing timestamps so lines break across gaps; they do not fill or modify data.
Return and distance panels show percent for readability; stored values remain fractions.
Volatility is per bar, not annualized. Rolling z-scores include the current bar and
need not have zero mean in this short window. The horizontal zero/0.5 lines are visual
references, not trading rules. No clipping, smoothing, or outlier removal is applied.


In [ ]:
def plot_grid(start, end):
    return pd.date_range(
        start, pd.Timestamp(end) + pd.Timedelta(days=1), freq="h", inclusive="left", tz="UTC"
    )


def feature_line(name, grid):
    return features[name].where(features[prefix + name]).reindex(grid)


grid = plot_grid(PLOT_START, PLOT_END)
fig, axes = plt.subplots(6, 1, figsize=(14, 17), sharex=True, constrained_layout=True)
bars["close"].reindex(grid).plot(ax=axes[0], color="black", label="BTC close")
axes[0].set_ylabel("USDT / BTC")
for name in ["close_over_mean_12", "close_over_mean_168"]:
    (100 * feature_line(name, grid)).plot(ax=axes[1], label=name)
axes[1].set_ylabel("Mean distance (%)")
for name in ["log_return_1", "log_return_24"]:
    (100 * feature_line(name, grid)).plot(ax=axes[2], label=name, alpha=0.75)
axes[2].set_ylabel("Log return (%)")
for name in ["volatility_24", "volatility_168", "parkinson_24", "garman_klass_24"]:
    feature_line(name, grid).plot(ax=axes[3], label=name, alpha=0.8)
axes[3].set_ylabel("Volatility / bar")
for name in ["volume_zscore_168", "dollar_volume_zscore_168"]:
    feature_line(name, grid).plot(ax=axes[4], label=name, alpha=0.8)
axes[4].set_ylabel("Rolling z-score")
feature_line("taker_buy_ratio", grid).plot(ax=axes[5], label="taker_buy_ratio")
axes[5].axhline(0.5, color="gray", linewidth=0.8, linestyle="--")
axes[5].set_ylabel("Taker buy fraction")
axes[5].set_ylim(0, 1)
for ax in axes[1:3].tolist() + [axes[4]]:
    ax.axhline(0, color="gray", linewidth=0.8)
for ax in axes:
    ax.legend(loc="upper right", fontsize=8, ncol=2)
axes[-1].set_xlabel("UTC candle open time")
fig.suptitle(f"{SYMBOL} {INTERVAL} — TRAIN — {PLOT_START} to {PLOT_END}")
fig.savefig(OUTPUT / "btc_features.png")
plt.show()

window = features.loc[PLOT_START:PLOT_END]
summary = window[names].describe().T[["count", "mean", "std", "min", "max"]]
summary["available_fraction_of_stored_rows"] = [window[prefix + n].mean() for n in names]
summary["units"] = specs["units"]
display(summary)

## Gap and availability inspection

This TRAIN window covers a source-data interruption. The heatmap distinguishes
unavailable stored features from absent hourly rows, and the close panel breaks at
absent candles. Longer lookbacks should recover later after a gap. Off-grid candles
are counted separately; they stay in the stored data but are not snapped onto the grid.
This view checks availability behavior, not market performance.


In [ ]:
gap_grid = plot_grid(GAP_START, GAP_END)
selected = ["log_return_1", "log_return_24", "volatility_24", "volume_zscore_168"]
gap_window = features.loc[GAP_START:GAP_END]
masks = gap_window[[prefix + n for n in selected]].reindex(gap_grid)
state = masks.astype("float64").to_numpy().T  # NaN: absent row, 0: unavailable, 1: usable
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, constrained_layout=True)
x = np.arange(len(gap_grid))
axes[0].plot(x, bars["close"].reindex(gap_grid), color="black")
axes[0].set_ylabel("USDT / BTC")

cmap = ListedColormap(["#e8a33e", "#287d8e"])
cmap.set_bad("#dedede")
axes[1].imshow(state, aspect="auto", interpolation="nearest", cmap=cmap, vmin=0, vmax=1)
axes[1].set_yticks(range(len(selected)), selected)
ticks = np.arange(0, len(gap_grid), 48)
axes[1].set_xticks(ticks, [gap_grid[i].strftime("%b %d") for i in ticks])
axes[1].set_xlabel("UTC — gray: absent hourly row; amber: unavailable; teal: available")
axes[1].grid(False)
fig.suptitle(f"{SYMBOL} TRAIN gap / lookback recovery — {GAP_START} to {GAP_END}")
fig.savefig(OUTPUT / "btc_availability.png")
plt.show()
off_grid = (gap_window["open_time"] % 3_600_000_000 != 0).sum()
missing = len(gap_grid.difference(gap_window.index))
print(f"Gap window: {missing} missing hourly timestamps; {off_grid} off-grid stored rows.")
print(f"Figures saved to {OUTPUT}")

## Review notes

Check that the price and mean-distance movements agree; returns oscillate around
zero with occasional large moves; volatility is nonnegative and responds to changing
movement; volume z-scores show bursts; taker-buy fraction stays within [0, 1]. Inspect
the gap view for missing rows and delayed recovery with longer lookbacks. A failed
assertion or hash check must be investigated before using the store.

These plots cannot establish predictive value or rule out every form of leakage.
Follow the accepted split, availability, cost and evaluation policies in later steps.
Re-run this notebook after rebuilding features; do not copy old observations into a
new result. No strategy parameters are selected here.
